# ArXiv Dataset Preprocessing
## Load dataset

In [9]:
import pandas as pd

df = pd.read_json("arxiv_ai_subset.json", lines=True)

print(df.head())

         id           submitter                           authors  \
0  704.0047         Igor Grabec            T. Kosel and I. Grabec   
1  704.0050         Igor Grabec            T. Kosel and I. Grabec   
2  704.0304   Carlos Gershenson                 Carlos Gershenson   
3  704.0671      Maxim Raginsky                    Maxim Raginsky   
4  704.0954  Jos\'e M. F. Moura  Soummya Kar and Jose M. F. Moura   

                                               title  \
0  Intelligent location of simultaneously active ...   
1  Intelligent location of simultaneously active ...   
2                  The World as Evolving Information   
3              Learning from compressed observations   
4  Sensor Networks with Random Links: Topology De...   

                                            comments  \
0          5 pages, 5 eps figures, uses IEEEtran.cls   
1          5 pages, 7 eps figures, uses IEEEtran.cls   
2  16 pages. Extended version, three more laws of...   
3  6 pages; submitted to

## Inspect columns

In [10]:
print(df.columns)

Index(['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi',
       'report-no', 'categories', 'license', 'abstract', 'versions',
       'update_date', 'authors_parsed'],
      dtype='object')


## Extract abstract column

In [11]:
df=df['abstract']
print(df.head)

<bound method NDFrame.head of 0           The intelligent acoustic emission locator is...
1           Part I describes an intelligent acoustic emi...
2           This paper discusses the benefits of describ...
3           The problem of statistical learning is to co...
4           In a sensor network, in practice, the commun...
                                ...                        
272612      Complex sequential decision-making planning ...
272613      Weight space symmetries in neural network ar...
272614    Models trained on real-world data often mirror...
272615      Laplacian learning method is a well-establis...
272616      Recently, there has been an extensive resear...
Name: abstract, Length: 272617, dtype: object>


## Check for null values

In [12]:
df.isnull().sum()

0

## Count duplicates

In [13]:
df.duplicated().sum()

86

## Drop duplicate rows

In [14]:
df.drop_duplicates(inplace=True)

## Confirm Series type

In [18]:
print(type(df))

<class 'pandas.core.series.Series'>


## Convert Series to DataFrame

In [19]:
df = df.to_frame(name="abstract")

## Filter abstracts longer than 20 words

In [20]:
df = df[df['abstract'].str.split().apply(len) > 20]

## Show DataFrame info

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 272410 entries, 0 to 272616
Data columns (total 1 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   abstract  272410 non-null  object
dtypes: object(1)
memory usage: 4.2+ MB


## Cast abstract to string type

In [24]:
df["abstract"] = df["abstract"].astype("string")

## Show updated info

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 272410 entries, 0 to 272616
Data columns (total 1 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   abstract  272410 non-null  string
dtypes: string(1)
memory usage: 4.2 MB


## Reset index

In [26]:
df = df.reset_index(drop=True)

## Add doc_id column

In [27]:
df["doc_id"] = ["arxiv_" + str(i) for i in range(len(df))]

## Build initial documents list

In [28]:
documents = []

for _, row in df.iterrows():
    documents.append({
        "doc_id": row["doc_id"],
        "text": row["abstract"]
    })

## Set up NLP preprocessing tools

In [30]:
import re
import nltk
from tqdm import tqdm

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

tqdm.pandas()

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\4m\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\4m\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\4m\AppData\Roaming\nltk_data...


## Define text cleaning function

In [31]:
def preprocess_text(text):
    
    # lowercase
    text = text.lower()
    
    # remove symbols
    text = re.sub(r'[^a-zA-Z0-9 ]', ' ', text)
    
    words = text.split()
    
    cleaned_words = []
    
    for word in words:
        if word not in stop_words:
            lemma = lemmatizer.lemmatize(word)
            cleaned_words.append(lemma)
    
    return " ".join(cleaned_words)

## Apply preprocessing to all abstracts

In [32]:
df["clean_text"] = df["abstract"].progress_apply(preprocess_text)

100%|██████████| 272410/272410 [15:09<00:00, 299.64it/s] 


## Filter short cleaned texts and reset index

In [33]:
df = df[df["clean_text"].str.split().apply(len) > 20]
df = df.reset_index(drop=True)

## Reassign doc_id after filtering

In [34]:
df["doc_id"] = ["arxiv_" + str(i) for i in range(len(df))]

## Preview cleaned text sample

In [35]:
df['clean_text'].head()

0    intelligent acoustic emission locator describe...
1    part describes intelligent acoustic emission l...
2    paper discusses benefit describing world infor...
3    problem statistical learning construct predict...
4    sensor network practice communication among se...
Name: clean_text, dtype: object

## Build final documents list with original and preprocessed text

In [36]:
documents = []

for _, row in df.iterrows():
    documents.append({
        "doc_id": row["doc_id"],
        "text": row["clean_text"],
        "original": row["abstract"]
    })

## Save documents list to JSON

In [38]:
import json

with open("arxiv_docs.json", "w") as f:
    json.dump(documents, f, indent=2)